# Task 1 — Merging Customer Social Profiles and Transactions

In this notebook, I explore and clean the two tabular datasets, then join customer social-profile information to transaction records. The merged file can be used as input for the product recommendation work in a later task; this notebook does not train a model.

## Before running

1. Open the [customer social profiles spreadsheet](https://docs.google.com/spreadsheets/d/10up-WdC0a6egYaXLKiMQUotpOvaKZRZvYuWFQx4-RPQ/edit?gid=862127784#gid=862127784) and export the relevant sheet as a CSV named `customer_social_profiles.csv`.
2. Open the [customer transactions spreadsheet](https://docs.google.com/spreadsheets/d/1s4WOVm49lmLQ8d9QbbbdgAcRTNh3m0KCH_5ciiaRZw0/edit?gid=1844409263#gid=1844409263) and export the relevant sheet as a CSV named `customer_transactions.csv`.
3. The exported files are expected in the `data` folder next to this notebook. If you saved them elsewhere, update `SOCIAL_PATH` and `TRANSACTIONS_PATH` in the setup cell. The spreadsheet links may require your course account.

In the supplied CSVs, the profile ID is formatted like `A123` and the transaction ID like `123`. This notebook uses the numeric portion as a shared customer key, after validating both formats. Please verify this mapping with the group before treating the merged data as final.

## 1. Import libraries and set file paths

I use pandas for tabular data cleaning and merging, and matplotlib/seaborn for exploratory plots. If a library is missing, install it in the notebook environment with `pip install pandas numpy matplotlib seaborn`.

In [5]:
from pathlib import Path
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid")
DATA_DIR = Path("data")
SOCIAL_PATH = DATA_DIR / "customer_social_profiles.csv"
TRANSACTIONS_PATH = DATA_DIR / "customer_transactions.csv"
OUTPUT_PATH = DATA_DIR / "merged_customer_transactions_social.csv"

print(f"Social profiles file: {SOCIAL_PATH.resolve()}")
print(f"Transactions file: {TRANSACTIONS_PATH.resolve()}")

Social profiles file: C:\Users\pc\yusuf_school\machineLearningPipeline_formative2MultimodalDataPreprocessingAssignment\data\customer_social_profiles.csv
Transactions file: C:\Users\pc\yusuf_school\machineLearningPipeline_formative2MultimodalDataPreprocessingAssignment\data\customer_transactions.csv


## 2. Load the CSV files and standardize column names

Standardizing headers makes later column comparisons easier. I keep the data values unchanged here and check for duplicate column names after normalization.

In [6]:
def normalize_column_name(column):
    """Convert a column label to lowercase snake_case for consistent matching."""
    name = str(column).strip().lower()
    name = re.sub(r"[^a-z0-9]+", "_", name)
    return name.strip("_")


def load_and_standardize_csv(path):
    """Read a CSV and normalize its headers, raising a useful error if absent."""
    if not path.is_file():
        raise FileNotFoundError(
            f"CSV file not found: {path.resolve()}\n"
            f"Current notebook working directory: {Path.cwd()}\n"
            "Export the source spreadsheet as CSV and save it at the path above, "
            "or set SOCIAL_PATH / TRANSACTIONS_PATH to the actual CSV locations."
        )
    frame = pd.read_csv(path)
    frame.columns = [normalize_column_name(column) for column in frame.columns]
    if frame.columns.duplicated().any():
        duplicates = frame.columns[frame.columns.duplicated()].tolist()
        raise ValueError(f"Header normalization created duplicate column names: {duplicates}")
    return frame


social = load_and_standardize_csv(SOCIAL_PATH)
transactions = load_and_standardize_csv(TRANSACTIONS_PATH)

print("Social profiles shape:", social.shape)
print("Transactions shape:", transactions.shape)

FileNotFoundError: Could not find data\customer_social_profiles.csv. Export the spreadsheet as CSV and place it there.

## 3. Initial data checks

I check a few rows, data types, descriptive statistics, missing values, exact duplicate rows, and columns shared by the two sources. The customer ID columns have different names, so I validate and standardize them in the merge-key step.

In [ ]:
for table_name, frame in [("Social profiles", social), ("Transactions", transactions)]:
    print(f"\n--- {table_name}: sample rows ---")
    display(frame.head())
    print(f"\n--- {table_name}: data types ---")
    display(frame.dtypes.rename("dtype").to_frame())
    print(f"\n--- {table_name}: descriptive statistics ---")
    display(frame.describe(include="all").T)
    print(f"\n--- {table_name}: missing values ---")
    display(frame.isna().sum().rename("missing_count").to_frame())
    print(f"Exact duplicate rows: {frame.duplicated().sum()}")

common_columns = sorted(set(social.columns).intersection(transactions.columns))
print("\nColumns found in both files:", common_columns)

## 4. Exploratory plots

The plots below are generated from numeric columns that actually exist in the CSVs. Their shapes and any visible outliers should be interpreted using the dataset's units and context; I do not assume any particular pattern before seeing the data.

### Plot 1 — Numeric feature distributions

Histograms show the spread and skew of numeric fields. A long tail or multiple peaks may be worth checking, but the plot alone does not explain why those values occur.

In [ ]:
numeric_fields = []
for table_name, frame in [("transactions", transactions), ("social", social)]:
    for column in frame.select_dtypes(include=np.number).columns:
        numeric_fields.append((table_name, column, frame[column].dropna()))

if numeric_fields:
    plot_fields = numeric_fields[: min(4, len(numeric_fields))]
    fig, axes = plt.subplots(1, len(plot_fields), figsize=(5 * len(plot_fields), 4))
    axes = np.atleast_1d(axes)
    for axis, (table_name, column, values) in zip(axes, plot_fields):
        sns.histplot(values, bins=20, kde=len(values) > 1, ax=axis)
        axis.set(title=f"{table_name}: {column}", xlabel=column, ylabel="Row count")
    plt.tight_layout()
    plt.show()
else:
    print("No numeric columns were detected, so a numeric distribution plot is not applicable.")

### Plot 2 — Numeric feature boxplots

Boxplots make extreme values easier to spot relative to the rest of each feature. Potential outliers should be investigated rather than removed automatically, because they may be valid purchases or customer values.

In [ ]:
if numeric_fields:
    plot_fields = numeric_fields[: min(4, len(numeric_fields))]
    fig, axes = plt.subplots(1, len(plot_fields), figsize=(5 * len(plot_fields), 4))
    axes = np.atleast_1d(axes)
    for axis, (table_name, column, values) in zip(axes, plot_fields):
        sns.boxplot(y=values, ax=axis)
        axis.set(title=f"{table_name}: {column}", xlabel="", ylabel=column)
    plt.tight_layout()
    plt.show()
else:
    print("No numeric columns were detected, so a numeric boxplot is not applicable.")

### Plot 3 — Numeric feature correlations

The heatmap shows pairwise correlations among numeric columns within one source table. Correlation is not causation, and fields from separate tables are not combined here because their row order may not represent the same customers.

In [ ]:
correlation_source_name = None
correlation_source = None
for table_name, frame in [("transactions", transactions), ("social profiles", social)]:
    numeric_frame = frame.select_dtypes(include=np.number)
    if numeric_frame.shape[1] >= 2:
        correlation_source_name = table_name
        correlation_source = numeric_frame
        break

if correlation_source is not None:
    correlation = correlation_source.corr()
    plt.figure(figsize=(max(6, 1.2 * len(correlation.columns)), max(4, 1.0 * len(correlation.columns))))
    sns.heatmap(correlation, annot=True, cmap="coolwarm", center=0, fmt=".2f")
    plt.title(f"Numeric feature correlations — {correlation_source_name}")
    plt.tight_layout()
    plt.show()
else:
    print("A correlation heatmap needs at least two numeric columns in the same source; none were available.")

## 5. Remove exact duplicate rows

I remove only rows that are exact duplicates across every column and report how many were removed. I do not remove rows just because one field is repeated; repeated transaction customers are expected in a transaction table.

In [ ]:
def drop_exact_duplicates(frame, table_name):
    """Drop fully identical rows and report how many rows were removed."""
    duplicate_count = int(frame.duplicated().sum())
    cleaned = frame.drop_duplicates().copy()
    print(f"{table_name}: removed {duplicate_count} exact duplicate row(s).")
    return cleaned


social = drop_exact_duplicates(social, "Social profiles")
transactions = drop_exact_duplicates(transactions, "Transactions")

## 6. Choose and check the merge key

I merge on a customer identifier shared by both files. If the automatic suggestion is missing or ambiguous, I inspect `common_columns` above and enter the correct normalized column name in `MERGE_KEY` before rerunning this cell.

In [ ]:
SOCIAL_ID_COLUMN = "customer_id_new"
TRANSACTION_ID_COLUMN = "customer_id_legacy"
MERGE_KEY = "_customer_id"

if SOCIAL_ID_COLUMN not in social.columns or TRANSACTION_ID_COLUMN not in transactions.columns:
    raise KeyError(
        "Expected customer ID columns were not found. "
        f"Social columns: {social.columns.tolist()}; transaction columns: {transactions.columns.tolist()}"
    )

# The files use A + digits for social IDs and digits for transaction IDs.
social_id_text = social[SOCIAL_ID_COLUMN].astype("string").str.strip()
transaction_id_text = transactions[TRANSACTION_ID_COLUMN].astype("string").str.strip()
social_key = social_id_text.str.extract(r"^A(\d+)$", expand=False)
transaction_key = transaction_id_text.str.extract(r"^(\d+)$", expand=False)

invalid_social_ids = social_id_text.notna() & social_key.isna()
invalid_transaction_ids = transaction_id_text.notna() & transaction_key.isna()
if invalid_social_ids.any() or invalid_transaction_ids.any():
    raise ValueError(
        "Customer IDs do not match the expected formats (social: A + digits; transactions: digits). "
        f"Invalid social IDs: {int(invalid_social_ids.sum())}; "
        f"invalid transaction IDs: {int(invalid_transaction_ids.sum())}."
    )

social[MERGE_KEY] = social_key
transactions[MERGE_KEY] = transaction_key
common_customer_ids = set(social[MERGE_KEY].dropna()).intersection(
    transactions[MERGE_KEY].dropna()
)

print("Profile ID column:", SOCIAL_ID_COLUMN)
print("Transaction ID column:", TRANSACTION_ID_COLUMN)
print("Matching rule: remove the leading A from profile IDs; compare the remaining digits.")
print("Distinct customer IDs shared by both files:", len(common_customer_ids))
print("Social-profile null keys:", int(social[MERGE_KEY].isna().sum()))
print("Transaction null keys:", int(transactions[MERGE_KEY].isna().sum()))
print("Repeated social-profile keys:", int(social[MERGE_KEY].duplicated(keep=False).sum()))
print("Repeated transaction keys (usually expected):", int(transactions[MERGE_KEY].duplicated(keep=False).sum()))

## 7. Validate and merge the two tables

I use transactions as the left table so every purchase row is retained. Since customers can have multiple social rows, I average their numeric scores and combine distinct category values before joining. This keeps all purchase rows and avoids a many-to-many row explosion. The output retains the original ID columns so the ID mapping can be reviewed.

In [ ]:
def combine_distinct_values(values):
    """Keep every distinct non-null category when summarizing a customer's profiles."""
    categories = sorted(values.dropna().astype(str).unique())
    return " | ".join(categories) if categories else pd.NA


# Exclude null lookup IDs; keep their transaction rows unmatched in the left join.
profiles_with_key = social.loc[social[MERGE_KEY].notna()].copy()
profile_feature_columns = [
    column for column in social.columns
    if column not in {MERGE_KEY, SOCIAL_ID_COLUMN}
]
profile_aggregations = {
    column: "mean" if pd.api.types.is_numeric_dtype(social[column]) else combine_distinct_values
    for column in profile_feature_columns
}
profiles_for_merge = profiles_with_key.groupby(MERGE_KEY).agg(profile_aggregations).reset_index()
profile_row_counts = profiles_with_key.groupby(MERGE_KEY).size().rename("social_profile_rows")
profiles_for_merge = profiles_for_merge.merge(
    profile_row_counts, on=MERGE_KEY, how="left", validate="one_to_one"
)
print("Profile rows excluded because the merge key is missing:", len(social) - len(profiles_with_key))
print("Unique profile customers after aggregation:", len(profiles_for_merge))
print("Profile customers with multiple source rows:", int((profile_row_counts > 1).sum()))

transaction_row_count = len(transactions)
merged = transactions.merge(
    profiles_for_merge,
    on=MERGE_KEY,
    how="left",
    validate="many_to_one",
    indicator=True,
    suffixes=("_transaction", "_social"),
)

if len(merged) != transaction_row_count:
    raise AssertionError(
        f"Expected {transaction_row_count} transaction rows after a left merge; got {len(merged)}."
    )

match_counts = merged["_merge"].value_counts()
matched_count = int(match_counts.get("both", 0))
unmatched_count = int(match_counts.get("left_only", 0))
match_rate = matched_count / len(merged) if len(merged) else 0.0

print("Transaction rows before merge:", transaction_row_count)
print("Rows after merge:", len(merged))
print("Matched transaction rows:", matched_count)
print("Unmatched transaction rows:", unmatched_count)
print(f"Transaction match rate: {match_rate:.1%}")
display(match_counts.rename("row_count").to_frame())

## 8. Review and save the merged dataset

I review the joined rows and missing-value counts, then save the result as a CSV. The `_merge` indicator is kept so matched and unmatched transactions can be checked later.

In [ ]:
print("Merged dataset shape:", merged.shape)
print("Merged columns:", merged.columns.tolist())
display(merged.head())

missing_after_merge = merged.isna().sum().sort_values(ascending=False)
display(missing_after_merge[missing_after_merge > 0].rename("missing_count").to_frame())

DATA_DIR.mkdir(parents=True, exist_ok=True)
merged.to_csv(OUTPUT_PATH, index=False)
print(f"Saved merged dataset to: {OUTPUT_PATH.resolve()}")

## 9. Merge validation checklist and takeaways

- The selected key exists in both source tables; review its meaning and type before relying on the result.
- Exact duplicate rows were removed, and conflicting duplicate profile IDs cause the merge to stop for manual review.
- A left join preserved the transaction row count; the match counts and missing-value summary show merge coverage.
- The output file is `data/merged_customer_transactions_social.csv`. Check `_merge` and the target product column before using the data in a later recommendation task.

A low match rate can indicate inconsistent customer IDs, different ID formatting, or customers with no social-profile row. I would investigate those cases rather than assume the join is correct.